Bibliotecas utilizadas

In [147]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import requests
import time

Configurações para acessar o navegador Chrome

In [148]:
def configurar_navegador():
    opcoes = Options()
    opcoes.add_argument('--ignore-certificate-errors') 
    opcoes.add_argument('--disable-blink-features=AutomationControlled') 
    
    driver = webdriver.Chrome(options=opcoes)
    return driver

driver = configurar_navegador()

Acessando o site da Embrapa

In [149]:
url = "https://agromet.cpact.embrapa.br/"
driver.get(url)

Tempo de espera (30 segundos)

In [150]:
wait = WebDriverWait(driver, 30)

Preparação para coleta dos dados

In [151]:
def acessar_frame_dados(driver):
    #Identificar o frame correto para acessar os dados
    driver.switch_to.default_content() 
    
    frames = driver.find_elements(By.TAG_NAME, "frame") + driver.find_elements(By.TAG_NAME, "iframe")
    
    for indice in range(len(frames)):
        driver.switch_to.default_content()
        driver.switch_to.frame(indice)
        
        try:
            wait_curto = WebDriverWait(driver, 3)
            wait_curto.until(EC.presence_of_element_located((By.ID, "lblRealV1")))
            return True 
            
        except:
            continue 

    return False

sucesso = acessar_frame_dados(driver)

Coleta dos dados

In [152]:
def extrair_dados_clima(driver):
    #Extrai os dados de clima do frame correto e monta o pacote para o n8n
    try:
        temperatura = driver.find_element(By.ID, "lblRealV1").text
        umidade = driver.find_element(By.ID, "lblRealV2").text
        sensacao = driver.find_element(By.ID, "lblRealV3").text 
        chuva = driver.find_element(By.ID, "lblAcumuladoV1").text

        print("🌤️ BOLETIM EMBRAPA CPACT")
        print("-" * 25)
        print(f"Temperatura: {temperatura}")
        print(f"Umidade: {umidade}")
        print(f"Sensação Térmica: {sensacao}")
        print(f"Chuva Diária: {chuva}\n")

        # 2. Cria o pacote (dicionário) que será enviado ao n8n
        dados = {
            "temperatura": temperatura,
            "umidade": umidade,
            "sensacao": sensacao,
            "chuva": chuva
        }
        return dados

    except Exception as e:
        print(f"Erro ao extrair os dados: {e}")
        return None

if sucesso: 
    boletim = extrair_dados_clima(driver)
    print("📦 Pacote 'boletim' criado e pronto para envio!")

🌤️ BOLETIM EMBRAPA CPACT
-------------------------
Temperatura: 16.5 °C
Umidade: 79 %
Sensação Térmica: 16.5 °C
Chuva Diária: 0.2 mm

📦 Pacote 'boletim' criado e pronto para envio!


Coletar os dados da cidade de Morro Redondo

In [153]:
def extrair_clima_morro_redondo():
    #Buscando os dados de clima de Morro Redondo via API pública da Open-Meteo
    # Coordenadas: Latitude -31.59, Longitude -52.64
    url = "https://api.open-meteo.com/v1/forecast?latitude=-31.59&longitude=-52.64&current=temperature_2m,relative_humidity_2m,apparent_temperature,precipitation"
    
    try:
        resposta = requests.get(url).json()
        atual = resposta["current"]
        
        dados_mr = {
            "temperatura_mr": f"{atual['temperature_2m']} °C",
            "umidade_mr": f"{atual['relative_humidity_2m']} %",
            "sensacao_mr": f"{atual['apparent_temperature']} °C",
            "chuva_mr": f"{atual['precipitation']} mm"
        }
        return dados_mr
        
    except Exception as e:
        print(f"Erro ao buscar Morro Redondo: {e}")
        return None

Enviar dados coletados para o n8n

In [154]:
def enviar_para_n8n(dados_clima):
    #Envia o pacote de dados para o n8n via webhook
    url_webhook = "http://localhost:5678/webhook/boletim-clima"
    
    print("🚀 Enviando dados reais para o n8n...")
    
    try:
        # Dispara o POST enviando o dicionário no formato JSON
        resposta = requests.post(url_webhook, json=dados_clima)
        
        # O código 200 significa "OK / Sucesso" na web
        if resposta.status_code == 200:
            print("✅ Sucesso Absoluto! O n8n recebeu o pacote.")
        else:
            print(f"⚠️ O dado foi, mas o n8n reclamou. Status: {resposta.status_code}")
            
    except Exception as e:
        print(f"❌ Erro de comunicação: {e}")

Juntando os dados das duas cidades

In [155]:
#Dados de Pelotas (via Embrapa)
boletim_pelotas = boletim 

#Dados de Morro Redondo (via API Open-Meteo)
boletim_morro_redondo = extrair_clima_morro_redondo()

pacote_final = {**boletim_pelotas, **boletim_morro_redondo}

#Envio para o n8n
enviar_para_n8n(pacote_final)

🚀 Enviando dados reais para o n8n...
✅ Sucesso Absoluto! O n8n recebeu o pacote.
